In [3]:
import os
import httpx
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma  
from langchain_core.prompts import PromptTemplate 
load_dotenv()

True

In [4]:
dummy_transcript = {
    "lang": "en",
    "availableLangs": ["en"],
    "content": [
        {
            "lang": "en",
            "text": "Welcome everyone. Today we're going to talk about machine learning fundamentals.",
            "offset": 0,
            "duration": 5200
        },
        {
            "lang": "en",
            "text": "Many beginners jump straight into deep learning without understanding the basics.",
            "offset": 5200,
            "duration": 4800
        },
        {
            "lang": "en",
            "text": "You should first understand linear regression, classification, and evaluation metrics.",
            "offset": 10000,
            "duration": 6100
        },
        {
            "lang": "en",
            "text": "Once you're comfortable with those concepts, neural networks become much easier to understand.",
            "offset": 16100,
            "duration": 6700
        },
        {
            "lang": "en",
            "text": "Let's now build our first model using Python and scikit-learn.",
            "offset": 22800,
            "duration": 5400
        },
        {
            "lang": "en",
            "text": "We'll split our dataset into training and testing sets before fitting the model.",
            "offset": 28200,
            "duration": 6200
        },
        {
            "lang": "en",
            "text": "Finally, we'll evaluate the model using accuracy, precision, recall, and F1 score.",
            "offset": 34400,
            "duration": 7000
        },
        {
            "lang": "en",
            "text": "In the next lecture we'll explore decision trees and random forests in more detail.",
            "offset": 41400,
            "duration": 6300
        }
    ]
}

In [5]:
def get_transcript(video_id: str) -> dict: #key is content  and content contains a list of dictionaries with keys as lang text offset duration
    # SUPADATA_API_KEY = os.environ.get("SUPADATA_API_KEY")
    
    # response = httpx.get(
    #     "https://api.supadata.ai/v1/youtube/transcript",
    #     params={"videoId": video_id},
    #     headers={"x-api-key": SUPADATA_API_KEY},
    #     timeout=60.0,
    # )
    
    # if response.status_code != 200:
    #     raise Exception(f"Supadata error: {response.status_code} - {response.text}")
    
    # data = response.json()
    
    # return data
    return dummy_transcript

data = get_transcript("WDSRXu4cJbM")

In [6]:
docs = []
current_text = ""
start_offset = None

for segment in data["content"]:
    if start_offset is None:
        start_offset = segment["offset"]

    current_text += " " + segment["text"]

    # create a chunk every 1000 characters
    if len(current_text) >= 100:
        docs.append(
            Document(
                page_content=current_text.strip(),
                metadata={"offset": start_offset}
            )
        )
        current_text = ""
        start_offset = None

# remaining text
if current_text:
    docs.append(
        Document(
            page_content=current_text.strip(),
            metadata={"offset": start_offset}
        )
    )

docs

[Document(metadata={'offset': 0}, page_content="Welcome everyone. Today we're going to talk about machine learning fundamentals. Many beginners jump straight into deep learning without understanding the basics."),
 Document(metadata={'offset': 10000}, page_content="You should first understand linear regression, classification, and evaluation metrics. Once you're comfortable with those concepts, neural networks become much easier to understand."),
 Document(metadata={'offset': 22800}, page_content="Let's now build our first model using Python and scikit-learn. We'll split our dataset into training and testing sets before fitting the model."),
 Document(metadata={'offset': 34400}, page_content="Finally, we'll evaluate the model using accuracy, precision, recall, and F1 score. In the next lecture we'll explore decision trees and random forests in more detail.")]

In [7]:
vector_store = Chroma(
    embedding_function=OpenAIEmbeddings(
        model="openai/text-embedding-3-large",
        dimensions=1536,
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url="https://openrouter.ai/api/v1"
    ),
    persist_directory='my_chroma_db',
    collection_name='videos_transcript_chunks_1536'
)


vector_store.add_documents(docs)

['ce09d528-02f1-4e88-bed1-ae7bb974eed9',
 '2ad43b74-0015-48fd-8829-2a9aba33ff4f',
 '82c55f18-3186-4ed2-a090-49a20bd21455',
 'd87f41d0-589c-43ae-b8f3-52f206aeaa70']

In [6]:
vector_store.similarity_search(
    query="where did he talk about linear regression and classification",
    k=1
)

[Document(id='b92038de-b550-4612-9926-91b5187128fb', metadata={'offset': 10000}, page_content="You should first understand linear regression, classification, and evaluation metrics. Once you're comfortable with those concepts, neural networks become much easier to understand.")]

In [22]:
retriever = vector_store.as_retriever(search_kwargs={"k" : 4})

In [23]:
# query = "where did he talk about precision and recall"
# result = retriever.invoke(query)
# result

In [24]:
llm = ChatOpenAI(
        model="openai/gpt-4o-mini",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url="https://openrouter.ai/api/v1"
)



In [25]:
# from langchain_core.prompts import ChatPromptTemplate
# prompt = ChatPromptTemplate.from_template("""
# You are answering questions about a YouTube video.

# Rules:
# - Answer only using the transcript context.
# - Do not make up facts.
# - If the answer is not in the transcript, say you couldn't find it.
# - Cite the timestamp(s) from the context whenever possible.
# - Keep the answer concise unless the user asks for more detail.

# Transcript Context:
# {context}
                                          
# Timestamp:
# {timestamp}

# User Question:
# {question}

# Answer:
# """)

In [26]:
# result

In [27]:
# context_text = "\n\n".join(doc.page_content for doc in result)
# context_text

# # context_timestamp = []
# # for doc in result:
# #     context_timestamp.append(doc.metadata['offset'])

# # context_timestamp

In [28]:
# final_prompt = prompt.invoke({"context": context_text, "question": query})
# final_prompt

In [29]:
# answer = llm.invoke(final_prompt)
# answer

In [30]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
# from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

In [31]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

def format_timestamp(retrieved_docs):
  context_timestamp = [doc.metadata["offset"] for doc in retrieved_docs]
  return context_timestamp


In [32]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough(),
    'timestamp' : retriever | RunnableLambda(format_timestamp)
})

In [33]:
# parser = StrOutputParser()
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

class output(BaseModel):

    content : str = Field(description='content to show')
    offset : List[int] = Field(description='used for timestamp')

parser = PydanticOutputParser(pydantic_object=output)

In [34]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
You are answering questions about a YouTube video.

Rules:
- Answer only using the transcript context.
- Do not make up facts.
- If the answer is not in the transcript, say you couldn't find it.
- Cite the timestamp(s) from the context whenever possible.
- Keep the answer concise unless the user asks for more detail.

Transcript Context:
{context}
                                          
Timestamp:
{timestamp}

User Question:
{question}

{format_instruction}
""",
partial_variables={'format_instruction' : parser.get_format_instructions()}
)

In [35]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
result = main_chain.invoke("where did he talk about precision and recall")

__main__.output